In [ ]:
# Install project in editable mode
!pip install -e ".[dev]"


## Shared library overview
This repository exposes a reusable package (`small_code_models`) with:
- `data.py`: dataset loading and `CloneDetectionDataset`
- `metrics.py`: consistent clone-detection metrics
- `trainer.py`: a thin Hugging Face trainer wrapper for reproducible runs


In [ ]:
import numpy as np
from transformers import AutoTokenizer
from small_code_models.data import CloneDetectionDataset
from small_code_models.metrics import compute_metrics

# Tiny synthetic dataset: 5 clone pairs + 5 non-clone pairs
pairs = [
    ("def add(a,b): return a+b", "def add(x,y): return x+y"),
    ("for i in range(10): print(i)", "for j in range(10): print(j)"),
    ("if x>0: x-=1", "if n>0: n-=1"),
    ("while n: n-=1", "while k: k-=1"),
    ("return sorted(nums)", "return sorted(values)"),
    ("def add(a,b): return a+b", "def multiply(a,b): return a*b"),
    ("print('hello')", "total = sum(items)"),
    ("if flag: run()", "class A: pass"),
    ("x = [i*i for i in xs]", "raise ValueError('bad')"),
    ("return min(nums)", "with open(p) as f: data=f.read()"),
]
labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

tokenizer = AutoTokenizer.from_pretrained('microsoft/codebert-base')
dataset = CloneDetectionDataset(tokenizer=tokenizer, pairs=pairs, labels=labels, max_length=128)
print('Dataset size:', len(dataset))

# Metric demo with toy logits
toy_logits = np.array([[0.2, 0.8]] * 5 + [[0.8, 0.2]] * 5)
metrics = compute_metrics((toy_logits, np.array(labels)))
metrics


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('microsoft/codebert-base', num_labels=2)
batch = tokenizer(pairs[0][0], pairs[0][1], return_tensors='pt', truncation=True, max_length=128)
outputs = model(**batch)
print('Logits shape:', outputs.logits.shape)
outputs.logits


## Switching to full benchmarks
Use any benchmark script with dataset and output directories:
```bash
python bcb_detection_models/codebert-bcb-01.py --data_dir /path/to/bcb --output_dir results/codebert_bcb
```
For complete reproduction across all datasets/models:
```bash
bash scripts/run_all_benchmarks.sh /path/to/datasets
```
